# Lektion 11 - Agent-till-Agent (A2A) Protokoll


## Installation


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Vad är A2A-protokollet?

**Agent-till-Agent (A2A) protokollet** är en öppen standard som möjliggör för AI-agenter att kommunicera,
upptäcka varandra och samarbeta — även när de är byggda på olika ramverk eller hostas
av olika tjänster.

Viktiga begrepp:

- **Upptäckt** – Agenter publicerar ett *Agentkort* som beskriver deras kapabiliteter, vilket gör det
  enkelt för andra agenter (eller orkestrerare) att hitta rätt specialist för en uppgift.
- **Meddelandepassning** – Agenter utbyter strukturerade meddelanden via ett gemensamt protokoll, så att en
  förfrågan från en agent kan förstås och utföras av en annan oavsett dess interna
  implementation.
- **Uppgiftslivscykel** – A2A definierar tillstånd som *inskickad*, *pågående*, *slutförd* och
  *misslyckad*, vilket ger orkestreraren full insyn i hur en delegerad uppgift fortskrider.

I denna lektion simulerar vi A2A-stil samarbete genom att koppla tre specialiserade reseagenter
in i ett arbetsflöde där varje agent bidrar med sin expertis och skickar resultat till nästa.


## Skapa specialiserade resebyråer


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Fleragentsamarbete via Arbetsflöde

Vi kopplar samman de tre agenterna till ett sekventiellt arbetsflöde som speglar A2A-meddelandeförmedling:

1. **CurrencyExchangeAgent** tar emot användarens förfrågan och producerar valutavägledning.
2. **ActivityPlannerAgent** tar emot den berikade kontexten och lägger till aktivitetsrekommendationer.
3. **TravelManagerAgent** syntetiserar båda inmatningarna till en slutgiltig reseöversikt.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Förstå A2A i produktion

I en produktionsmiljö låser A2A-protokollet upp kraftfulla scenarier för tjänst-till-tjänst-kommunikation:

| Funktion | Beskrivning |
|---|---|
| **Interop mellan ramverk** | En agent som är byggd med ett ramverk kan delegera uppgifter till en agent byggd med vilket annat A2A-kompatibelt ramverk som helst, vilket möjliggör verklig interoperabilitet över organisationer. |
| **Tjänstegränser** | Agenter kan existera i separata mikrotjänster, molnregioner eller till och med olika organisationer samtidigt som de samarbetar sömlöst. |
| **Dynamisk upptäckt** | En orkestrator kan vid körning fråga en Agent Card-registry för att hitta den bäst lämpade specialisten för en given deluppgift. |
| **Strömning & push-notifikationer** | A2A stödjer Server-Sent Events (SSE) för uppdateringar i realtid och push-notifikationer för långvariga uppgifter. |

Arbetsflödet vi byggde ovan är en förenklad, in-process-version av detta mönster. I en verklig
distribution skulle varje agent exponera en HTTP-endpoint, publicera en Agent Card och kommunicera
via A2A JSON-RPC-protokollet.


## Sammanfattning

I denna lektion lärde du dig:

1. **Vad A2A-protokollet är** — en öppen standard för agent-till-agent upptäckt, meddelandehantering,
   och uppgiftshantering.
2. **Hur man skapar specialiserade agenter** — en valutaväxlingsagent, en aktivitetsplanerare,
   och en resehanteringsorkestrator.
3. **Hur man kopplar ihop agenter i ett arbetsflöde** — med `WorkflowBuilder` för att modellera sekventiell
   meddelandehantering mellan agenter.
4. **Hur A2A fungerar i produktion** — möjliggör samarbete mellan ramverk och tjänster
   med dynamisk upptäckt och strömmande uppdateringar.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
